# 05 — XGBoost Modeling, Ablation & Comparison

**Owner:** Jivit Rana · **Phase 1, Step 5**

This notebook trains the gradient-boosting benchmark for the Water Quality Prediction project (CSAI Project 18) on the shared, area-grouped split, using `src/models_xgb.py`.

### Methodological Standards
1. **Regime 1 (Formula Emulation / Benchmark Surrogate)**: The 8 chemical inputs mathematically define `CCME_Values`, so a high score here measures how well XGBoost can *imitate the CCME formula*. It is **not** independent environmental forecasting.
2. **Fair-comparison rules**:
   - One shared split with a fixed seed (`get_splits()`).
   - Hyperparameter tuning and early stopping use the **validation** set only.
   - The **test** set is scored exactly once, at the end.
3. **Same function format as Random Forest**: `train()` in `src/models_xgb.py` mirrors `src/models_rf.py`, so the Phase 2 MLP can plug into the same evaluation.

In [1]:
import sys
sys.path.insert(0, '..')

import warnings
import pandas as pd

from src.config import REPORTS, SEED, TARGET_COLUMN
from src.models_xgb import train, run_ablation

warnings.filterwarnings('ignore', category=UserWarning)
pd.set_option('display.width', 200)

print(f"Seed: {SEED} | Target: {TARGET_COLUMN}")

Seed: 42 | Target: CCME_Values


## 1. Theoretical Foundation: Gradient Boosting

Gradient boosting (Friedman, 2001) builds a model as a **sum of small trees**, added one at a time:
$$F_m(x) = F_{m-1}(x) + \eta \, h_m(x)$$

### 1.1 Loss function
We minimise squared error, $L(y, F) = \tfrac{1}{2}(y - F(x))^2$. Each new tree $h_m$ is fitted to the **negative gradient** of the loss at the current model:
$$r_{im} = -\frac{\partial L(y_i, F_{m-1}(x_i))}{\partial F_{m-1}(x_i)} = y_i - F_{m-1}(x_i)$$
For squared error this is simply the **residual**: every new tree learns what the ensemble still gets wrong.

### 1.2 Additive trees
Each $h_m$ is a shallow tree, weak on its own. Adding many of them lets later trees correct earlier mistakes step by step. This differs from Random Forest, which averages *independent* deep trees to cut variance.

### 1.3 Shrinkage
The learning rate $\eta \in (0, 1]$ scales every tree down, so no single tree changes the prediction much. A small $\eta$ needs more trees but generalises better.

### 1.4 Regularization (Chen & Guestrin, 2016)
XGBoost minimises a regularised objective:
$$\mathcal{L} = \sum_i L(y_i, \hat{y}_i) + \sum_k \Omega(h_k), \qquad \Omega(h) = \gamma T + \tfrac{1}{2}\lambda \sum_j w_j^2 + \alpha \sum_j |w_j|$$
where $T$ is the number of leaves and $w_j$ the leaf values. `reg_lambda` ($\lambda$) and `reg_alpha` ($\alpha$) shrink leaf values, `min_child_weight` blocks tiny leaves, and `subsample` / `colsample_bytree` train each tree on random rows and columns. XGBoost also uses the **second derivative** of the loss to choose better splits.

### 1.5 Early stopping
We allow up to 2,000 trees and stop when validation RMSE has not improved for 50 rounds. The validation set picks the number of trees; the test set plays no part.

## 2. Hyperparameter Tuning and Training (Regime 1, 8/8, Engineered Features)

We search with **Optuna** (TPE sampler, 15 trials) over learning rate, depth, subsample, column sampling, minimum child weight, and L1/L2 regularization.

### Strict Scientific Guarantee
- **Training partition**: fits the trees.
- **Validation partition**: scores every Optuna trial and drives early stopping.
- **Test partition**: not used during the search.

Each trial trains on a 300,000-row sample for speed. The best settings are then refitted on the **full training set** (about 2.0M rows).

In [2]:
res = train(
    regime=1,
    feature_set="engineered",
    partial_scenario="8/8",
    do_tune=True,
    n_trials=15,
    save_reports=True,   # appends to reports/xgb_results.csv
)

print(f"=== XGBoost Results ({res['config']['experiment_label']}) ===")
print(f"Training rows: {res['config']['n_train_samples']:,} | Features: {res['config']['n_features']}")
print(f"Trees used (early stopping): {res['best_iteration']}")
print("\nBest hyperparameters:")
for name, value in res["best_params"].items():
    print(f"  {name:18s} {value:.4f}" if isinstance(value, float) else f"  {name:18s} {value}")

=== XGBoost Results (surrogate_emulator_benchmark (formula approximation, NOT independent prediction)) ===
Training rows: 1,986,397 | Features: 23
Trees used (early stopping): 1991

Best hyperparameters:
  learning_rate      0.0552
  max_depth          7
  subsample          0.9502
  colsample_bytree   0.8140
  min_child_weight   30
  reg_alpha          0.0071
  reg_lambda         33.1164


## 3. Ablation: Raw vs Engineered Features

We compare XGBoost **with and without** Himanshu's engineered features on the identical split:
1. **Raw**: the 8 chemical measurements plus `Country` and `Waterbody Type` (10 features).
2. **Engineered**: raw features plus stoichiometric ratios, log transforms and cyclic time features (23 features).

Both arms use the **same hyperparameters** (the tuned values above), the same seed and the same split, so any score difference comes from the features alone.

In [3]:
ablation = run_ablation(
    regime=1,
    partial_scenario="8/8",
    params=res["best_params"],
    save_reports=False,
)

gain = 1 - ablation.loc[1, "test_rmse"] / ablation.loc[0, "test_rmse"]
print("=== Ablation: Raw vs Engineered (identical split & hyperparameters) ===")
print(ablation.to_string(index=False))
print(f"\nEngineered features change test RMSE by {-gain:+.1%}")
print(f"Saved to: {REPORTS / 'xgb_ablation.csv'}")

=== Ablation: Raw vs Engineered (identical split & hyperparameters) ===
 regime partial_scenario feature_set  n_features  best_iteration  val_rmse  test_mae  test_rmse  test_r2
      1              8/8         raw          10            1999  0.424070  0.151689   0.400473 0.999539
      1              8/8  engineered          23            1991  0.354872  0.099990   0.328160 0.999690

Engineered features change test RMSE by -18.1%
Saved to: /Users/jivitrana/Desktop/water_quality_prediction/reports/xgb_ablation.csv


## 4. Random Forest vs XGBoost (Same Split)

We compare against Himanshu's tuned-baseline Random Forest, read from `reports/rf_results.csv` (Regime 1, 8/8, engineered features).

**Caveat:** that Random Forest was trained on a 100,000-row subsample, while XGBoost above used the full training set. Part of any gap therefore comes from data size, not only from the algorithm.

In [4]:
rf_all = pd.read_csv(REPORTS / "rf_results.csv")
rf = rf_all[(rf_all.regime == 1) & (rf_all.partial_scenario == "8/8") & (rf_all.feature_set == "engineered")].iloc[-1]

comparison = pd.DataFrame([
    {"Model": "Random Forest", "Train rows": int(rf.n_train_samples), "Test MAE": rf.test_mae, "Test RMSE": rf.test_rmse, "Test R²": rf.test_r2},
    {"Model": "XGBoost", "Train rows": res["config"]["n_train_samples"],
     "Test MAE": res["test_metrics"]["mae"], "Test RMSE": res["test_metrics"]["rmse"], "Test R²": res["test_metrics"]["r2"]},
])
print(comparison.to_string(index=False))

        Model  Train rows  Test MAE  Test RMSE  Test R²
Random Forest      100000  0.407842   0.919717 0.997568
      XGBoost     1986397  0.099990   0.328160 0.999690


### 4.1 Why one wins

- **Random Forest** averages many *independent* deep trees. This cuts variance, but every tree must approximate the whole CCME formula by itself, so the averaged model keeps some bias.
- **XGBoost** fits each new tree to the **residual** of the ensemble, so error is removed step by step (bias falls), while shrinkage and regularization keep variance in check.
- The target is a deterministic, piecewise-smooth function of the inputs with almost no noise. That is the setting where residual fitting works best, so boosting is expected to win.
- **Limit of this claim:** the Random Forest saw 100k rows and XGBoost the full set, so the table shows the combined effect of algorithm and data size.

## 5. Final Holdout Evaluation & Export

The final XGBoost model was scored **once** on the holdout test partition (415,804 rows) inside `train()`, after all tuning choices were frozen. The result was appended to `reports/xgb_results.csv`.

In [5]:
final = pd.DataFrame([{
    "Model": "XGBoost (8/8 Engineered, tuned)",
    "Regime": "1 (Formula Emulation)",
    "Train Samples": res["config"]["n_train_samples"],
    "Features": res["config"]["n_features"],
    "Val RMSE": res["val_metrics"]["rmse"],
    "Val R²": res["val_metrics"]["r2"],
    "Test MAE": res["test_metrics"]["mae"],
    "Test RMSE": res["test_metrics"]["rmse"],
    "Test R²": res["test_metrics"]["r2"],
}])

print("=== Final Selected Model: Holdout Test Generalization ===")
print(final.to_string(index=False))
print(f"\nResults appended to: {REPORTS / 'xgb_results.csv'}")

=== Final Selected Model: Holdout Test Generalization ===
                          Model                Regime  Train Samples  Features  Val RMSE   Val R²  Test MAE  Test RMSE  Test R²
XGBoost (8/8 Engineered, tuned) 1 (Formula Emulation)        1986397        23  0.354872 0.999634   0.09999    0.32816  0.99969

Results appended to: /Users/jivitrana/Desktop/water_quality_prediction/reports/xgb_results.csv
